# Adult Income Dataset — Cleaning, Preprocessing & Feature Engineering (Beginner Guide)

This notebook is designed for learning, not just running code. Read each explanation before executing the following cell.

**Dataset:** `adult_income.csv` (put this file in the same folder as the notebook).

**Your assignment has two different targets:**
- **Question 2 / regression:** predict `hours-per-week`.
- **Questions 4–5 / classification:** predict whether `income` is `>50K`.

We will build clean, separate input/target tables for those tasks. We will not train models in this notebook yet.


## 1. Load the data and understand its shape

Each row represents one record; each column represents an attribute. The Adult/Census Income dataset is a benchmark dataset originally collected from 1994 census data and is used to study whether annual income exceeds $50,000. The original dataset has 48,842 records and 14 input attributes plus the income target. See the [UCI dataset page](https://archive.ics.uci.edu/dataset/2/adult).

`skipinitialspace=True` removes spaces after commas. `na_values=['?']` tells pandas that `?` means missing rather than a genuine category.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "adult_income.csv"

df_raw = pd.read_csv(
    DATA_PATH,
    skipinitialspace=True,
    na_values=["?"]
)

print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())
df_raw.info()


Shape: (48842, 15)
Columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       46043 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      46033 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  47985 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


## 2. Standardize text values and income labels

Some rows can have leading/trailing spaces. The combined Adult files can also have income labels like `<=50K.` and `>50K.`. They mean the same classes as `<=50K` and `>50K`, so normalize them once.

We keep a separate `df_raw` as an untouched backup and clean a copy named `df`.


In [2]:
df = df_raw.copy()

# Strip extra whitespace from text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Normalize punctuation variants in the income label
df["income"] = df["income"].str.rstrip(".")

print("Income labels:")
print(df["income"].value_counts(dropna=False))
print("\nShape after text normalization:", df.shape)


Income labels:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Shape after text normalization: (48842, 15)


## 3. Missing-value audit

Never assume that missing values only appear as blank cells. In this file, `?` encodes missingness. After loading with `na_values=['?']`, the expected missing values are mainly in `workclass`, `occupation`, and `native-country`.

Missing values do not automatically mean a row is useless. For this assignment, we can impute missing input features inside a pipeline instead of deleting every row.


In [3]:
missing = df.isna().sum().sort_values(ascending=False)
missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(2)
})
display(missing_table[missing_table["missing_count"] > 0])


,missing_count,missing_percent
occupation,2809,5.75
workclass,2799,5.73
native-country,857,1.75


## 4. Duplicate rows: inspect before deleting

After whitespace and income-label normalization, this copy of the CSV has some exact duplicate rows. An exact duplicate may be a duplicated record, but in a dataset without a person ID, it can also represent two people whose recorded attributes happen to be identical. Therefore, count duplicates and make a deliberate decision rather than deleting them automatically.

For this learning workflow, we **keep them by default**. If your instructor asks you to remove exact duplicate records, you can use the optional line shown below and document that assumption.


In [4]:
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows after normalization:", duplicate_count)

# OPTIONAL only if you decide these are erroneous duplicate records:
# df = df.drop_duplicates().copy()


Exact duplicate rows after normalization: 52


## 5. Validate values and inspect categories

Before transforming data, check category counts and whether numeric values have plausible ranges. A value that looks unusual is not necessarily wrong. The Adult data commonly has ages in the adult range, education levels represented by integers 1–16, non-negative capital gain/loss, and hours per week between 1 and 99.


In [5]:
print("Numeric summary:")
display(df.describe().T)

print("Categorical cardinality and most common values:")
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}: {df[col].nunique(dropna=True)} non-missing categories")
    print(df[col].value_counts(dropna=False).head(8))

# Basic range checks (these are checks, not automatic corrections)
range_checks = {
    "age": (1, 110),
    "education-num": (1, 16),
    "capital-gain": (0, None),
    "capital-loss": (0, None),
    "hours-per-week": (1, 168),
    "fnlwgt": (1, None),
}
for col, (low, high) in range_checks.items():
    bad = df[col] < low
    if high is not None:
        bad = bad | (df[col] > high)
    print(f"{col}: {int(bad.sum())} rows outside chosen sanity range")


Numeric summary:


,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


Categorical cardinality and most common values:

workclass: 8 non-missing categories
workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
NaN                  2799
State-gov            1981
Self-emp-inc         1695
Federal-gov          1432
Without-pay            21
Name: count, dtype: int64

education: 16 non-missing categories
education
HS-grad         15784
Some-college    10878
Bachelors        8025
Masters          2657
Assoc-voc        2061
11th             1812
Assoc-acdm       1601
10th             1389
Name: count, dtype: int64

marital-status: 7 non-missing categories
marital-status
Married-civ-spouse       22379
Never-married            16117
Divorced                  6633
Separated                 1530
Widowed                   1518
Married-spouse-absent      628
Married-AF-spouse           37
Name: count, dtype: int64

occupation: 14 non-missing categories
occupation
Prof-specialty       6172
Craft-repair         6112
Exec-managerial    


income: 2 non-missing categories
income
<=50K    37155
>50K     11687
Name: count, dtype: int64
age: 0 rows outside chosen sanity range
education-num: 0 rows outside chosen sanity range
capital-gain: 0 rows outside chosen sanity range
capital-loss: 0 rows outside chosen sanity range
hours-per-week: 0 rows outside chosen sanity range
fnlwgt: 0 rows outside chosen sanity range


## 6. Should we merge rare categories?

**Default decision: do not manually merge categories yet.** `native-country` has many categories and many are uncommon, but one-hot encoding 40-ish categories is manageable for this dataset. A rare category can still have a different income relationship. Merge categories only when you have a reason:

- categories are semantically similar and can be grouped defensibly;
- a category has so few training examples that estimates are unreliable;
- you compare a grouped-category approach against an ungrouped baseline on validation data.

`OneHotEncoder(handle_unknown='ignore')` handles categories that appear at prediction time but were absent during training. An optional `min_frequency=50` setting can automatically group infrequent categories; try it as a comparison rather than assuming it must be better.


## 7. `education` vs `education-num`: they duplicate information

In this CSV, each `education` category maps to one `education-num`, and each `education-num` maps back to one education category. They contain the same underlying education-level information in two representations.

Keep both available while doing EDA, because you may want to compare them. For a model, keep **one** by default:

- Keep `education` and one-hot encode it if you want to allow separate effects for each level without assuming equal numeric steps.
- Or keep `education-num` as numeric if you want a compact, ordered representation, accepting the numeric/ordinal assumption for models that use it.

For the recommended modeling pipeline below, we keep `education` and drop `education-num`.


In [6]:
edu_to_num = df.groupby("education")["education-num"].nunique(dropna=True)
num_to_edu = df.groupby("education-num")["education"].nunique(dropna=True)
print("Max distinct education-num values per education label:", int(edu_to_num.max()))
print("Max distinct education labels per education-num:", int(num_to_edu.max()))

# If both outputs are 1, the two columns are one-to-one on this dataset.


Max distinct education-num values per education label: 1
Max distinct education labels per education-num: 1


## 8. What about `fnlwgt`?

`fnlwgt` means **final weight** and is a census sampling weight, not an individual's body weight or salary. It can matter for population-representative statistical analysis. For this assignment's individual-level prediction exercise, dropping it is a defensible choice because it is not an ordinary personal attribute. Document this choice.

If a rubric explicitly expects every original predictor, run an additional baseline with `fnlwgt` retained and compare. For population estimates, do not simply treat it like an ordinary personal feature; consider appropriate survey/sample-weight methods.


## 9. Outliers: do not blindly cap every IQR flag

An IQR rule marks values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`. It is a flag for inspection, not proof of an error.

This dataset illustrates why automatic capping can be harmful:
- `capital-gain` and `capital-loss` have Q1 = Q3 = 0, so IQR = 0. The IQR rule would flag every positive value and a capper could turn informative gains/losses into zero.
- `hours-per-week` has many values above 52.5 because its middle 50% is concentrated near 40–45 hours. Many 55–99 hour records may be genuine. For Question 2, `hours-per-week` is the target, so do not cap or scale it as an input feature.
- Higher `age` values and some education levels flagged by an IQR rule can still be valid.

**Recommended baseline: no global outlier capping.** Check data validity and use transformations for highly skewed predictors as an optional feature-engineering experiment.


In [7]:
numeric_cols_all = df.select_dtypes(include=np.number).columns
rows = []
for col in numeric_cols_all:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    flagged = ((s < low) | (s > high)).sum()
    rows.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_fence": low,
        "upper_fence": high,
        "IQR_flag_count": int(flagged),
        "IQR_flag_percent": round(100 * flagged / len(s), 2)
    })
display(pd.DataFrame(rows).set_index("feature"))


,Q1,Q3,IQR,lower_fence,upper_fence,IQR_flag_count,IQR_flag_percent
feature,,,,,,,
age,28.0,48.0,20.0,-2.00,78.00,216,0.44
fnlwgt,117550.5,237642.0,120091.5,-62586.75,417779.25,1453,2.97
education-num,9.0,12.0,3.0,4.50,16.50,1794,3.67
capital-gain,0.0,0.0,0.0,0.00,0.00,4035,8.26
capital-loss,0.0,0.0,0.0,0.00,0.00,2282,4.67
hours-per-week,40.0,45.0,5.0,32.50,52.50,13496,27.63


## 10. Optional feature engineering for capital gains/losses

Capital gain and capital loss are highly right-skewed with many zero values. One experiment is to represent **whether the value is positive** separately from its **magnitude**:

- `has_capital_gain` = 1 if capital gain > 0, otherwise 0;
- `capital_gain_log1p` = `log(1 + capital-gain)`;
- analogous features for capital loss.

`log1p` compresses very large amounts but preserves zero as zero. This is an optional engineered variant, not a mandatory cleaning step. When using this variant, drop the raw capital-gain/loss columns from that model's feature matrix so you can compare the engineered representation against a baseline. The transformation is per-row and uses no information learned from the test set.


In [8]:
df_fe = df.copy()
for raw_col, stem in [("capital-gain", "capital_gain"), ("capital-loss", "capital_loss")]:
    df_fe[f"has_{stem}"] = (df_fe[raw_col] > 0).astype("int8")
    df_fe[f"{stem}_log1p"] = np.log1p(df_fe[raw_col])

# Inspect the engineered features
display(df_fe[["capital-gain", "has_capital_gain", "capital_gain_log1p",
               "capital-loss", "has_capital_loss", "capital_loss_log1p"]].head())


,capital-gain,has_capital_gain,capital_gain_log1p,capital-loss,has_capital_loss,capital_loss_log1p
0,2174,1,7.684784,0,0,0.0
1,0,0,0.000000,0,0,0.0
2,0,0,0.000000,0,0,0.0
3,0,0,0.000000,0,0,0.0
4,0,0,0.000000,0,0,0.0


## 11. Define deliberate feature-selection decisions

For the recommended, easy-to-explain baseline:
- drop `fnlwgt` (sampling weight; see explanation above);
- drop `education-num` and keep `education` (redundant representations);
- do not add `high_income` to the dataframe; make the binary classification target separately when needed;
- do not include the current task's target among input features.

For Question 2, we keep `income` as an input because the assignment says to predict hours using the other available columns. In a real deployment, include it only if income category would genuinely be known at the moment you need to predict hours. A feature unavailable at prediction time should not be used.


In [9]:
DROP_COMMON = ["fnlwgt", "education-num"]

# Question 2: regression target is hours-per-week
regression_target = "hours-per-week"
X_reg = df.drop(columns=[regression_target] + DROP_COMMON)
y_reg = df[regression_target].copy()

print("Regression X shape:", X_reg.shape)
print("Regression y shape:", y_reg.shape)
print("Regression input columns:", X_reg.columns.tolist())

# Optional alternative: if income would not be known when predicting hours,
# uncomment the following to exclude it from regression inputs:
# X_reg = X_reg.drop(columns=["income"])


Regression X shape: (48842, 12)
Regression y shape: (48842,)
Regression input columns: ['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'native-country', 'income']


## 12. Split before fitting learned preprocessing steps

We split the raw input features and target first. The imputer learns medians/missing-category handling from the training partition; the scaler learns training means and standard deviations; the encoder learns categories from training data. The test partition receives those same learned settings.


In [10]:
from sklearn.model_selection import train_test_split

valid_target = y_reg.notna()
X_reg = X_reg.loc[valid_target].copy()
y_reg = y_reg.loc[valid_target].copy()

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=42
)

print(X_reg_train.shape, X_reg_test.shape)


(39073, 12) (9769, 12)


## 13. Build numerical and categorical pipelines

Numerical inputs:
1. fill any missing values with the training-set median;
2. standardize numerical scales.

Categorical inputs:
1. preserve missingness as a `Missing` category;
2. one-hot encode categories;
3. safely ignore unseen test categories.

For this Adult CSV, actual missing values mainly occur in categorical columns, but retaining numerical imputation makes the pipeline more robust if the data changes.


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_cols = X_reg_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_reg_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numerical columns: ['age', 'capital-gain', 'capital-loss']
Categorical columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']


## 14. Fit the preprocessing only; no model training yet

`fit_transform` fits the transformations on training data and transforms it. `transform` applies those same learned transformations to the test data. The output may be a sparse matrix because one-hot encoding typically creates many zeros; that is normal.


In [12]:
X_reg_train_processed = preprocessor_reg.fit_transform(X_reg_train)
X_reg_test_processed = preprocessor_reg.transform(X_reg_test)

print("Processed train shape:", X_reg_train_processed.shape)
print("Processed test shape:", X_reg_test_processed.shape)
print("Target remains on original scale:", y_reg_train.head().tolist())


Processed train shape: (39073, 106)
Processed test shape: (9769, 106)
Target remains on original scale: [40, 40, 40, 20, 70]


## 15. Optional: group infrequent categories

You asked whether you should merge category options. Start without merging. If you want an experiment later, use `min_frequency=50` in the encoder. Categories with fewer than 50 training examples are grouped into an infrequent bucket (where supported by your scikit-learn version). Evaluate this against the default encoder; do not assume grouping improves performance.

Replace the encoder step with:

```python
OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50)
```

If your scikit-learn version does not support `min_frequency` or `infrequent_if_exist`, stick to `OneHotEncoder(handle_unknown="ignore")`.


## 16. Prepare the classification target for Questions 4–5

For income classification, `income` becomes the target and must be removed from `X`. We use 1 for `>50K` and 0 for `<=50K`. Never include `high_income` or `income` itself in the classification inputs because that would reveal the answer.


In [13]:
# Use the same feature-selection decisions, but change the target
classification_target = "income"
y_cls = df[classification_target].map({"<=50K": 0, ">50K": 1})

if y_cls.isna().any():
    raise ValueError("Unexpected or missing income labels. Inspect df['income'].")

X_cls = df.drop(columns=[classification_target] + DROP_COMMON)

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls,
    test_size=0.20,
    random_state=42,
    stratify=y_cls
)

num_cls = X_cls_train.select_dtypes(include=np.number).columns.tolist()
cat_cls = X_cls_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_cls = ColumnTransformer([
    ("numeric", numeric_pipeline, num_cls),
    ("categorical", categorical_pipeline, cat_cls)
])

X_cls_train_processed = preprocessor_cls.fit_transform(X_cls_train)
X_cls_test_processed = preprocessor_cls.transform(X_cls_test)

print("Classification train/test processed:", X_cls_train_processed.shape, X_cls_test_processed.shape)
print("Training target proportions:")
print(y_cls_train.value_counts(normalize=True).round(3))


Classification train/test processed: (39073, 106) (9769, 106)
Training target proportions:
income
0    0.761
1    0.239
Name: proportion, dtype: float64


## 17. Final decisions checklist

Before training models, make sure you can explain each item:

- [ ] `?` is interpreted as missing; whitespace and trailing periods in income labels are normalized.
- [ ] Missingness and exact duplicates are inspected, not blindly erased.
- [ ] `education` and `education-num` are redundant in this dataset, so a modeling input set keeps one.
- [ ] `fnlwgt` is a final/sampling weight and is dropped from the default individual-prediction feature set by documented choice.
- [ ] Rare categories are not merged by default; any grouping is treated as an experiment.
- [ ] No global IQR capping is used by default, because it would flag legitimate values and all positive capital gains/losses when IQR = 0.
- [ ] Preprocessing is fitted on training inputs only.
- [ ] Question 2 uses `hours-per-week` as `y`; Question 4/5 uses `income` as `y`.
- [ ] The active target is never included in its own input features.

**Next step:** after these checks, train Linear Regression and SGD Regressor with `Pipeline([('preprocessor', preprocessor_reg), ('model', estimator)])`. For income classification, create a separate pipeline ending with Logistic Regression or KNN. Keep the test set fixed for final comparisons.
